# 03 select hvg8000

**Purpose:** Select 8,000 highly variable genes while preserving raw counts.

**Original analysis notebook:** `03_HVG_core_4datasets_HVG8000_lowmem_Xcounts_corrected.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# HVG8000 low-memory branch for scVI

This notebook selects 8000 HVGs from the metadata-standardized full-gene object. It is optimized for lower RAM usage on a 16 GB RAM machine.

Important design choice: the output pre-scVI object stores raw counts in `.X` only. It does not duplicate the same matrix into `layers['counts']`, because duplicating a 351k × 8000 matrix can crash VS Code/Jupyter. Downstream scVI should therefore use `layer=None` and treat `.X` as raw counts.


In [ ]:
# =========================
# Low-memory imports and runtime limits
# =========================

import os

# Reduce accidental CPU thread over-subscription, which can increase memory pressure.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")

from pathlib import Path
import gc
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp

warnings.filterwarnings("default")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
LOGS_DIR = PROJECT_DIR / "logs"
INTEGRATED_DIR = PROCESSED_DIR / "integrated"
for d in [PROCESSED_DIR, METADATA_DIR, FIGURES_DIR, LOGS_DIR, INTEGRATED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_FULLGENE_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad"
FALLBACK_FULLGENE_H5AD = INTEGRATED_DIR / "GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad"

HVG_N_TOP = 8000
HVG_BATCH_KEY = "core_dataset"

OUTPUT_HVG_H5AD = INTEGRATED_DIR / f"GBM_core_4datasets_HVG{HVG_N_TOP}_pre_scVI_lowmem_Xcounts.h5ad"
OUTPUT_HVG_ALIAS = INTEGRATED_DIR / f"GBM_core_HVG{HVG_N_TOP}_pre_scVI_lowmem_Xcounts.h5ad"

HVG_TABLE_TSV = METADATA_DIR / f"core_4datasets_HVG{HVG_N_TOP}_table_lowmem.tsv"
HVG_SELECTED_GENES_TSV = METADATA_DIR / f"core_4datasets_HVG{HVG_N_TOP}_selected_genes_lowmem.tsv"
HVG_INPUT_INTEGRITY_TSV = METADATA_DIR / f"core_4datasets_HVG{HVG_N_TOP}_input_integrity_lowmem.tsv"
HVG_SUMMARY_TSV = METADATA_DIR / f"core_4datasets_HVG{HVG_N_TOP}_summary_lowmem.tsv"
CELL_COUNTS_TSV = METADATA_DIR / f"core_4datasets_HVG{HVG_N_TOP}_cell_counts_by_dataset_condition_lowmem.tsv"
HVG_OVERLAP_BY_DATASET_TSV = METADATA_DIR / f"core_4datasets_HVG{HVG_N_TOP}_overlap_by_dataset_lowmem.tsv"


def replace_path(path: Path):
    path = Path(path)
    if path.exists():
        path.unlink()
        print(f"[REPLACED existing file] {path}")


def write_tsv_replace(df: pd.DataFrame, path: Path, index=False):
    replace_path(path)
    df.to_csv(path, sep="\t", index=index)
    print(f"[SAVED] {path}")


def write_h5ad_replace(adata, path: Path):
    replace_path(path)
    adata.write_h5ad(path, compression="gzip")
    print(f"[SAVED] {path}")

print("HVG_N_TOP:", HVG_N_TOP)
print("Primary input exists:", INPUT_FULLGENE_H5AD.exists(), INPUT_FULLGENE_H5AD)
print("Fallback input exists:", FALLBACK_FULLGENE_H5AD.exists(), FALLBACK_FULLGENE_H5AD)
print("Output:", OUTPUT_HVG_H5AD)


In [ ]:
# =========================
# Read full-gene input and validate integrity
# =========================

if INPUT_FULLGENE_H5AD.exists():
    FULLGENE_INPUT = INPUT_FULLGENE_H5AD
elif FALLBACK_FULLGENE_H5AD.exists():
    FULLGENE_INPUT = FALLBACK_FULLGENE_H5AD
    print("[WARNING] Metadata-standardized object not found. Using fallback merged object.")
else:
    raise FileNotFoundError(
        "No input full-gene h5ad was found. Expected one of:\n"
        f"1) {INPUT_FULLGENE_H5AD}\n"
        f"2) {FALLBACK_FULLGENE_H5AD}"
    )

adata = sc.read_h5ad(FULLGENE_INPUT)
print("Loaded:", adata)
print("Input file:", FULLGENE_INPUT)
print("n cells:", adata.n_obs)
print("n genes:", adata.n_vars)
print("layers:", list(adata.layers.keys()))

if adata.n_obs == 0:
    raise ValueError("Input object has zero cells.")
if adata.n_vars < 10000:
    raise ValueError(f"Input object has only {adata.n_vars} genes; expected full-gene/common-gene object.")
if "counts" not in adata.layers:
    raise KeyError("adata.layers['counts'] is missing. Raw counts are required.")
if adata.obs_names.duplicated().sum() > 0:
    raise ValueError("Duplicate cell barcodes found.")
if adata.var_names.duplicated().sum() > 0:
    raise ValueError("Duplicate gene symbols found.")
if adata.layers["counts"].shape != adata.X.shape:
    raise ValueError(f"counts layer shape {adata.layers['counts'].shape} does not match X shape {adata.X.shape}.")

required_obs_cols = ["core_dataset", "dataset_id", "geo_id", "sample_id", "patient_id", "condition", "scvi_batch"]
missing_required = [c for c in required_obs_cols if c not in adata.obs.columns]
if missing_required:
    raise KeyError("Missing required obs columns: " + ", ".join(missing_required))

for col in required_obs_cols:
    adata.obs[col] = adata.obs[col].astype(str).fillna("unknown")
    adata.obs[col] = adata.obs[col].replace({"nan": "unknown", "None": "unknown", "": "unknown"})

# Minimal count integrity check without materializing dense arrays.
counts = adata.layers["counts"]
if sp.issparse(counts):
    sample_values = counts.data[: min(20000, counts.data.size)]
else:
    sample_values = np.asarray(counts).ravel()[:20000]
integer_like = bool(np.allclose(sample_values, np.round(sample_values))) if sample_values.size else True
nonnegative = bool(np.all(sample_values >= 0)) if sample_values.size else True

integrity_rows = [
    {"field": "input_fullgene_h5ad", "value": str(FULLGENE_INPUT)},
    {"field": "n_cells", "value": int(adata.n_obs)},
    {"field": "n_genes", "value": int(adata.n_vars)},
    {"field": "has_counts_layer", "value": True},
    {"field": "counts_integer_like_sample", "value": integer_like},
    {"field": "counts_nonnegative_sample", "value": nonnegative},
    {"field": "output_hvg_h5ad", "value": str(OUTPUT_HVG_H5AD)},
    {"field": "memory_mode", "value": "lowmem_X_counts_no_duplicate_counts_layer"},
]
for col in required_obs_cols:
    integrity_rows.append({
        "field": col,
        "value": f"n_unique={adata.obs[col].nunique()}; n_unknown={(adata.obs[col].astype(str) == 'unknown').sum()}"
    })

integrity = pd.DataFrame(integrity_rows)
write_tsv_replace(integrity, HVG_INPUT_INTEGRITY_TSV)
display(integrity)


In [ ]:
# =========================
# Cell composition report
# =========================

cell_counts = (
    adata.obs
    .groupby(["core_dataset", "condition", "sample_id", "patient_id"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values(["core_dataset", "condition", "sample_id", "patient_id"])
)
write_tsv_replace(cell_counts, CELL_COUNTS_TSV)
display(cell_counts.head(20))

print("Cell counts by core_dataset:")
display(adata.obs["core_dataset"].value_counts().rename_axis("core_dataset").reset_index(name="n_cells"))
print("Cell counts by condition:")
display(adata.obs["condition"].value_counts().rename_axis("condition").reset_index(name="n_cells"))


In [ ]:
# =========================
# Highly variable genes: no full adata.copy()
# =========================

# Work directly on adata.var to avoid copying the 351k x 15k matrix.
# HVG uses counts layer and writes statistics into adata.var.

gene_sum = np.asarray(adata.layers["counts"].sum(axis=0)).ravel()
keep_nonzero_genes = gene_sum > 0
n_zero_genes = int((~keep_nonzero_genes).sum())
print("Zero-count genes:", n_zero_genes)

if n_zero_genes > 0:
    # This creates a smaller view/copy only if necessary. Usually it should not happen after merge QC.
    adata = adata[:, keep_nonzero_genes].copy()
    print("After removing zero-count genes:", adata)

if adata.n_vars < HVG_N_TOP:
    raise ValueError(f"Requested {HVG_N_TOP} HVGs but only {adata.n_vars} genes are available.")

print("HVG batch key:", HVG_BATCH_KEY)
print(adata.obs[HVG_BATCH_KEY].value_counts())

hvg_method_used = None
hvg_error = None

try:
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=HVG_N_TOP,
        flavor="seurat_v3",
        layer="counts",
        batch_key=HVG_BATCH_KEY,
        subset=False,
    )
    hvg_method_used = "seurat_v3_counts_batch_core_dataset_lowmem"
    print("[HVG] seurat_v3 completed successfully.")
except Exception as e:
    hvg_error = repr(e)
    raise RuntimeError(
        "seurat_v3 HVG failed. In low-memory mode I intentionally do not run the log-normalized fallback, "
        "because that fallback can materialize large temporary matrices and crash a 16 GB RAM system. "
        f"Original error: {hvg_error}"
    )

if "highly_variable" not in adata.var.columns:
    raise KeyError("HVG selection finished but adata.var['highly_variable'] was not created.")

n_hvg = int(adata.var["highly_variable"].sum())
print("HVG selected:", n_hvg)

hvg_table = adata.var.copy()
hvg_table.insert(0, "gene", hvg_table.index.astype(str))
hvg_table["selected_hvg"] = hvg_table["highly_variable"].astype(bool)
hvg_table["hvg_method_used"] = hvg_method_used
hvg_table["hvg_batch_key"] = HVG_BATCH_KEY
write_tsv_replace(hvg_table, HVG_TABLE_TSV)

selected = hvg_table[hvg_table["selected_hvg"]].copy()
sort_cols = [c for c in ["highly_variable_rank", "highly_variable_nbatches", "means"] if c in selected.columns]
if sort_cols:
    ascending_map = {"highly_variable_rank": True, "highly_variable_nbatches": False, "means": False}
    selected = selected.sort_values(sort_cols, ascending=[ascending_map[c] for c in sort_cols])
write_tsv_replace(selected, HVG_SELECTED_GENES_TSV)
display(selected.head(20))


In [ ]:
# =========================
# Create low-memory HVG object and save
# =========================

hvg_mask = adata.var["highly_variable"].astype(bool).values
selected_genes = adata.var_names[hvg_mask].astype(str).tolist()
print("selected genes:", len(selected_genes))

# Build a minimal AnnData object.
# Critical low-memory choice: X contains raw counts, and we do NOT duplicate X into layers['counts'].
X_hvg = adata.layers["counts"][:, hvg_mask].copy()
obs_hvg = adata.obs.copy()
var_hvg = adata.var.loc[selected_genes].copy()

adata_hvg = ad.AnnData(X=X_hvg, obs=obs_hvg, var=var_hvg)

# Convert common metadata to categorical to reduce size and make scVI setup cleaner.
for col in ["core_dataset", "dataset_id", "geo_id", "study_id", "sample_id", "patient_id", "condition", "diagnosis", "tumor_status", "scvi_batch"]:
    if col in adata_hvg.obs.columns:
        adata_hvg.obs[col] = adata_hvg.obs[col].astype("category")

adata_hvg.uns["hvg_selection"] = {
    "source_fullgene_h5ad": str(FULLGENE_INPUT),
    "output_hvg_h5ad": str(OUTPUT_HVG_H5AD),
    "n_top_genes_requested": int(HVG_N_TOP),
    "n_hvg_selected": int(adata_hvg.n_vars),
    "hvg_method_used": str(hvg_method_used),
    "hvg_batch_key": str(HVG_BATCH_KEY),
    "x_in_output": "raw counts",
    "counts_layer_in_output": "not stored to avoid duplicating memory; use scVI layer=None",
    "memory_mode": "lowmem_X_counts_no_duplicate_counts_layer",
    "note": "This is the HVG/scVI branch. Full-gene object remains the source for marker DE and CNV."
}

write_h5ad_replace(adata_hvg, OUTPUT_HVG_H5AD)
write_h5ad_replace(adata_hvg, OUTPUT_HVG_ALIAS)

print("Final low-memory HVG object:", adata_hvg)
print("layers:", list(adata_hvg.layers.keys()))
print("X is raw counts for scVI:", True)
print("n cells:", adata_hvg.n_obs)
print("n HVGs:", adata_hvg.n_vars)

# Release large temporary objects that are no longer needed in this notebook.
del X_hvg
gc.collect()


In [ ]:
# =========================
# Final reports
# =========================

summary_rows = [
    {"metric": "input_fullgene_h5ad", "value": str(FULLGENE_INPUT)},
    {"metric": "output_hvg_h5ad", "value": str(OUTPUT_HVG_H5AD)},
    {"metric": "output_hvg_compatibility_alias", "value": str(OUTPUT_HVG_ALIAS)},
    {"metric": "n_cells", "value": int(adata_hvg.n_obs)},
    {"metric": "n_genes_input_after_zero_filter", "value": int(adata.n_vars)},
    {"metric": "n_hvg_selected", "value": int(adata_hvg.n_vars)},
    {"metric": "hvg_method_used", "value": hvg_method_used},
    {"metric": "hvg_batch_key", "value": HVG_BATCH_KEY},
    {"metric": "has_counts_layer", "value": bool("counts" in adata_hvg.layers)},
    {"metric": "x_contains_raw_counts", "value": True},
    {"metric": "x_shape", "value": str(adata_hvg.X.shape)},
    {"metric": "memory_mode", "value": "lowmem_X_counts_no_duplicate_counts_layer"},
    {"metric": "scvi_setup_layer_should_be", "value": "None"},
    {"metric": "n_core_datasets", "value": int(adata_hvg.obs["core_dataset"].nunique()) if "core_dataset" in adata_hvg.obs.columns else "missing"},
    {"metric": "n_samples", "value": int(adata_hvg.obs["sample_id"].nunique()) if "sample_id" in adata_hvg.obs.columns else "missing"},
    {"metric": "n_patients", "value": int(adata_hvg.obs["patient_id"].nunique()) if "patient_id" in adata_hvg.obs.columns else "missing"},
]
summary = pd.DataFrame(summary_rows)
write_tsv_replace(summary, HVG_SUMMARY_TSV)
display(summary)

cc = (
    adata_hvg.obs
    .groupby(["core_dataset", "condition"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values(["core_dataset", "condition"])
)
display(cc)

overlap_rows = []
if "highly_variable_nbatches" in adata.var.columns:
    selected_all = adata.var[adata.var["highly_variable"]].copy()
    nbatch_counts = selected_all["highly_variable_nbatches"].value_counts().sort_index()
    for nb, n in nbatch_counts.items():
        overlap_rows.append({"highly_variable_nbatches": int(nb), "n_selected_genes": int(n)})
else:
    overlap_rows.append({"highly_variable_nbatches": "not_available_for_method", "n_selected_genes": int(adata_hvg.n_vars)})

overlap = pd.DataFrame(overlap_rows)
write_tsv_replace(overlap, HVG_OVERLAP_BY_DATASET_TSV)
display(overlap)

assert adata_hvg.n_obs == adata.n_obs, "Cell number changed unexpectedly during HVG selection."
assert adata_hvg.n_vars == HVG_N_TOP or adata_hvg.n_vars > 0, "No HVGs were selected."
assert "counts" not in adata_hvg.layers, "Low-memory output should not duplicate counts layer."
print("[DONE] HVG low-memory notebook completed successfully.")
